In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import json

with open("chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Number of chunks:", len(chunks))

Number of chunks: 3225


In [3]:
texts = [chunk["text"] for chunk in chunks]

print("Number of texts:", len(texts))

Number of texts: 3225


In [4]:
embeddings = model.encode(
    texts,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches: 100%|██████████| 101/101 [00:28<00:00,  3.52it/s]

Embedding shape: (3225, 384)


In [5]:
import numpy as np

np.save("embeddings.npy", embeddings)

print("Saved embeddings.npy")

Saved embeddings.npy


In [6]:
# testing 
embeddings = np.load("embeddings.npy")

In [7]:
query = "What are the different approaches for evaluating an agent?"

In [8]:
query_embedding = model.encode(query)

In [9]:
from sentence_transformers import SentenceTransformer, util
scores = util.cos_sim(
    query_embedding,
    embeddings
)[0]

In [12]:
scores

tensor([0.3577, 0.6300, 0.6450,  ..., 0.0111, 0.0635, 0.0382])

In [10]:
top_k = 5

top_indices = scores.argsort(descending=True)[:top_k]

In [13]:
top_indices

tensor([   2, 1294, 1293,    1,    4])

In [11]:
for rank, idx in enumerate(top_indices, start=1):

    chunk = chunks[idx]

    print("=" * 80)
    print(f"Rank: {rank}")
    print(f"Score: {scores[idx].item():.4f}")
    print(f"Page: {chunk['metadata']['page_title']}")
    print(f"Section: {chunk['metadata']['section']}")
    print()
    print(chunk["text"])
    print()

Rank: 1
Score: 0.6450
Page: Agent Evaluation
Section: 2. Human Annotation

A human reviews the agent's output and labels/scores it.
For example:
Humans can evaluate things that are difficult to capture with code, such as tone, usefulness, safety, and overall quality.
Pros: High-quality ground truth
Cons: Expensive, slow, and potentially subjective.

Rank: 2
Score: 0.6450
Page: AI Roadmap
Section: 2. Human Annotation

A human reviews the agent's output and labels/scores it.
For example:
Humans can evaluate things that are difficult to capture with code, such as tone, usefulness, safety, and overall quality.
Pros: High-quality ground truth
Cons: Expensive, slow, and potentially subjective.

Rank: 3
Score: 0.6300
Page: AI Roadmap
Section: 1. LLM-as-a-Judge

Use another LLM to evaluate the agent's output.
Give the input + agent response + evaluation criteria to a judge LLM.
Judge scores things like:
Correctness
Relevance
Helpfulness
Reasoning quality
Tool usage
Can use a 1–5 score, pass/fa